# A3 multimodal EDA — Amazon Berkeley Objects

Group NNs · M1 · 18 November 2026. This notebook reproduces the EDA from the downloaded source and fixed manifests. The target is the native catalog tag. The dataset decision is pending. Code-cell outputs below come from actual Python execution.

In [1]:
from pathlib import Path
import hashlib
import json
import subprocess
import sys

repo = Path.cwd()
if not (repo / 'scripts/a3_eda.py').exists():
    repo = repo.parent
completed = subprocess.run(
    [sys.executable, str(repo / 'scripts/a3_eda.py')],
    cwd=repo, check=True, capture_output=True, text=True
)
print('Reproduced the raw-source audit, figures and saved review summary.')

Reproduced the raw-source audit, figures and saved review summary.


In [2]:
summary = json.loads((repo / 'results/a3/eda/a3_eda_summary.json').read_text())
keys = ['selected_pairs', 'classes', 'train_validation_test', 'raw_missingness',
        'word_length_quantiles', 'image_short_side_quantiles', 'label_flag_counts',
        'pair_review_counts', 'label_review_counts', 'versions']
print(json.dumps({key: summary[key] for key in keys}, indent=2, ensure_ascii=False))

{
  "selected_pairs": 10627,
  "classes": 12,
  "train_validation_test": {
    "test": 1063,
    "train": 8501,
    "validation": 1063
  },
  "raw_missingness": {
    "raw_all": {
      "records": 147702,
      "missing_text_any_language": 0,
      "no_english_title": 24968,
      "missing_main_image_id": 575,
      "missing_either_modality": 575,
      "missing_or_ambiguous_label": 0,
      "rates_percent": {
        "missing_text_any_language": 0.0,
        "no_english_title": 16.90431,
        "missing_main_image_id": 0.3893,
        "missing_either_modality": 0.3893,
        "missing_or_ambiguous_label": 0.0
      }
    },
    "raw_selected_classes": {
      "records": 27316,
      "missing_text_any_language": 0,
      "no_english_title": 6924,
      "missing_main_image_id": 23,
      "missing_either_modality": 23,
      "missing_or_ambiguous_label": 0,
      "rates_percent": {
        "missing_text_any_language": 0.0,
        "no_english_title": 25.34778,
        "missing_main_ima

## Class distribution

![Fixed split class counts](../results/a3/eda/a3_class_distribution.png)

## Title lengths

![Word and character distributions](../results/a3/eda/a3_text_length_distribution.png)

## Image geometry

![Image short side and aspect ratio](../results/a3/eda/a3_image_geometry.png)

## Pair and label review

Every selected item/image/title/label mapping is checked against its native archive record. A fixed-seed review of 36 pairs found compatible product families, nine SHOES examples depicting sandals and one SOFA fabric swatch. Fine product attributes were not exhaustively checked. The mixed review sample is qualitative evidence. Original tags are retained for the catalog-classification proposal.

![Examples of native label breadth](../results/a3/eda/a3_proposal_pair_examples.jpg)

Review notes are in `results/a3/eda/a3_pair_review.csv`. Automated title flags are review candidates and include complete chairs sold with cushions.

In [3]:
for filename, expected in summary['manifest_sha256'].items():
    path = repo / 'data/processed/abo' / filename
    actual = hashlib.sha256(path.read_bytes()).hexdigest()
    assert actual == expected, filename
assert summary['raw_missingness']['selected_source_pairs_checked'] == summary['selected_pairs']
assert summary['raw_missingness']['selected_source_pair_mismatches'] == 0
assert summary['selected_missing_text'] == 0
assert summary['selected_missing_image_files'] == 0
print('Manifest hashes and selected native-pair checks passed.')
print(json.dumps(summary['split_validation'], indent=2))

Manifest hashes and selected native-pair checks passed.
{
  "cross_split_item_overlap": 0,
  "cross_split_group_overlap": 0,
  "cross_split_image_id_overlap": 0,
  "cross_split_pixel_overlap": 0,
  "cross_split_normalized_title_overlap": 0,
  "phash_grouping_hamming_radius": 4,
  "all_manifest_images_verified": true
}


## Split and interpretation

Connected product/image/title/model/perceptual groups stay in one split. The largest selected group has 1,211 pairs, reflecting conservative footwear grouping. Titles may strongly signal the catalog tag, and native categories can overlap. Source, cleaning and EDA hashes identify this custom subset. The proposal and experimental plan are in `reports/a3/a3_proposal.md` and its PDF export.